# MaizeFormerX — Maize Leaf Disease Classification

**Paper:** MaizeFormerX: a lightweight vision transformer with cross-scale attention for explainable maize leaf disease diagnosis
**Journal:** Scientific Reports (2026) — DOI: 10.1038/s41598-026-44550-0
**Framework:** PyTorch

## Architecture Overview (confirmed from GitHub: github.com/rezaul-h/MaizeFormerX/)
- **MultiScalePatchEmbed:** patch_sizes=[8,16,32], strides=[8,16,32], embed_dims=[64,96,128] (each scale has its own dim)
- **CSAF:** projects each scale to fusion_dim=192, concatenates all tokens → [B,1029,192], self-attention (4 heads, residual=True)
- **Transformer Encoder Stack:** 6 layers, 6 heads, dim=192, mlp_ratio=4.0, drop_path linear schedule 0→0.1
- **Classifier:** CLS token → Dropout(0.3) → Linear(192, num_classes)
- **Total params:** ~9.6M

## Token counts (IMG_SIZE=224)
| Scale | Patch/Stride | N tokens | Embed dim |
|-------|-------------|---------|-----------|
| Fine  | 8×8         | 28²=784 | 64        |
| Mid   | 16×16       | 14²=196 | 96        |
| Coarse| 32×32       | 7²=49   | 128       |
| After CSAF concat | — | 784+196+49=1029 | 192 |
| After prepend CLS | — | 1030    | 192       |

## Experiment Design
- **Scenario 1 (Baseline):** Resize + ToTensor + Normalize only (no augmentation)
- **Scenario 2 (Augmented):** + Rotation(±30°) + RandomResizedCrop(scale=0.9–1.1) + HorizontalFlip + ColorJitter

## Key Hyperparameters (from paper)
| Param | Value |
|-------|-------|
| Optimizer | Adam |
| Learning Rate | 5e-5 |
| Weight Decay | 5e-5 |
| Batch Size | 32 |
| Max Epochs | 30 |
| Dropout | 0.3 |
| LR Schedule | CosineAnnealingLR |
| Warmup Steps | 300 |
| Early Stopping Patience | 4 |
| Input Size | 224×224 |


## BLOCK 1: Install Dependencies

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q torchinfo pytorch-grad-cam openpyxl Pillow

## BLOCK 2: Imports

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import math
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, recall_score
)
from sklearn.utils.class_weight import compute_class_weight
from torchinfo import summary

print("All imports successful.")

## BLOCK 3: Config & Reproducibility

In [ ]:

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
import torch
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device + Multi-GPU (Rule 8) ───────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()
print(f"Device: {device} | GPUs available: {n_gpus}")

# GPU determinism (Rule 9)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Hyperparameters (from paper Table 4) ─────────────────────────────────────
IMG_SIZE     = 224
BATCH_SIZE   = 16
LR           = 5e-5
WEIGHT_DECAY = 5e-5
DROPOUT      = 0.3
MAX_EPOCHS   = 30
WARMUP_STEPS = 300
PATIENCE     = 4

# ── Model Architecture (confirmed from GitHub: github.com/rezaul-h/MaizeFormerX/) ─
PATCH_SIZES       = [8, 16, 32]
EMBED_DIMS        = [64, 96, 128]
FUSION_DIM        = 192
NUM_LAYERS        = 6
NUM_HEADS_ENCODER = 6
NUM_HEADS_CSAF    = 4
MLP_RATIO         = 4.0
DROP_PATH         = 0.1
ATTN_DROPOUT      = 0.1
PROJ_DROPOUT      = 0.1

# ── Dataset path ─────────────────────────────────────────────────────────────
DATASET_PATH = "/kaggle/input/datasets/tminhhi/ot-mixed/dataset_smartcrop_300"

TRAIN_RATIO = 0.75
VAL_RATIO   = 0.15
TEST_RATIO  = 0.10

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

print(f"Config loaded.")
print(f"  DATASET_PATH={DATASET_PATH}")
print(f"  PATCH_SIZES={PATCH_SIZES}, EMBED_DIMS={EMBED_DIMS}, FUSION_DIM={FUSION_DIM}")
print(f"  NUM_LAYERS={NUM_LAYERS}, NUM_HEADS_ENCODER={NUM_HEADS_ENCODER}, NUM_HEADS_CSAF={NUM_HEADS_CSAF}")
print(f"  DROP_PATH={DROP_PATH}, ATTN_DROPOUT={ATTN_DROPOUT}, PROJ_DROPOUT={PROJ_DROPOUT}")
print(f"  BATCH_SIZE={BATCH_SIZE} (reduced from 32 for T4 OOM fix), LR={LR}, MAX_EPOCHS={MAX_EPOCHS}")


## BLOCK 4: Dataset Split Strategy

In [ ]:
# ── Dataset Split ─────────────────────────────────────────────────────────────
# Paper split: 80% train / 5% val / 15% test
# Notebook uses 75/15/10 since user dataset may differ from paper datasets

_TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
_VAL_DIR   = os.path.join(DATASET_PATH, 'val')
_VALID_DIR = os.path.join(DATASET_PATH, 'valid')
_TEST_DIR  = os.path.join(DATASET_PATH, 'test')

# ── Auto-detect number of classes ────────────────────────────────────────────
def get_num_classes_and_names():
    """Auto-detect NUM_CLASSES and class_names from dataset folder structure."""
    for d in [_TRAIN_DIR, _VALID_DIR, _VAL_DIR, DATASET_PATH]:
        if os.path.isdir(d):
            classes = sorted([
                c for c in os.listdir(d)
                if os.path.isdir(os.path.join(d, c))
            ])
            if classes:
                return len(classes), classes
    raise RuntimeError(f"Cannot find class folders in {DATASET_PATH}")

NUM_CLASSES, CLASS_NAMES_GLOBAL = get_num_classes_and_names()
print(f"NUM_CLASSES={NUM_CLASSES}")
print(f"Classes: {CLASS_NAMES_GLOBAL}")


def make_splits(transform_train, transform_eval):
    """
    Returns (train_ds, val_ds, test_ds) based on Kaggle dataset folder structure.
    Priority: 3 folders > train+valid > single train folder.
    """
    has_test  = os.path.isdir(_TEST_DIR)
    has_val   = os.path.isdir(_VAL_DIR)
    has_valid = os.path.isdir(_VALID_DIR)

    if has_val and has_test:
        print("Split mode: 3 folders (train / val / test)")
        return (
            datasets.ImageFolder(_TRAIN_DIR, transform=transform_train),
            datasets.ImageFolder(_VAL_DIR,   transform=transform_eval),
            datasets.ImageFolder(_TEST_DIR,  transform=transform_eval)
        )

    if has_valid and not has_test:
        print("Split mode: train/valid -> val+test split from valid (50/50)")
        full_eval = datasets.ImageFolder(_VALID_DIR, transform=transform_eval)
        n = len(full_eval)
        n_val = n // 2
        n_test = n - n_val
        val_ds, test_ds = random_split(
            full_eval, [n_val, n_test],
            generator=torch.Generator().manual_seed(SEED)
        )
        return (
            datasets.ImageFolder(_TRAIN_DIR, transform=transform_train),
            val_ds, test_ds
        )

    # Single train folder: random split
    print(f"Split mode: single train folder -> random split "
          f"{TRAIN_RATIO:.0%}/{VAL_RATIO:.0%}/{TEST_RATIO:.0%}")
    full_ds = datasets.ImageFolder(_TRAIN_DIR)
    n = len(full_ds)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    n_test  = n - n_train - n_val
    train_idx, val_idx, test_idx = random_split(
        range(n), [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(SEED)
    )
    train_ds = Subset(
        datasets.ImageFolder(_TRAIN_DIR, transform=transform_train),
        train_idx.indices
    )
    val_ds = Subset(
        datasets.ImageFolder(_TRAIN_DIR, transform=transform_eval),
        val_idx.indices
    )
    test_ds = Subset(
        datasets.ImageFolder(_TRAIN_DIR, transform=transform_eval),
        test_idx.indices
    )
    return train_ds, val_ds, test_ds


def get_class_names(dataset):
    """Auto-detect class_names from ImageFolder or Subset."""
    if hasattr(dataset, 'classes'):
        return dataset.classes
    elif hasattr(dataset, 'dataset') and hasattr(dataset.dataset, 'classes'):
        return dataset.dataset.classes
    return CLASS_NAMES_GLOBAL

## BLOCK 5: MaizeFormerX Architecture

In [ ]:
# ── MaizeFormerX Architecture (confirmed from GitHub: github.com/rezaul-h/MaizeFormerX/) ──
#
# Architecture flow:
#   Input [B, 3, 224, 224]
#     -> MultiScalePatchEmbed -> [[B,784,64], [B,196,96], [B,49,128]]
#     -> CSAF (project each to 192, concat, self-attn) -> [B, 1029, 192]
#     -> prepend CLS token -> [B, 1030, 192]
#     -> TransformerEncoder (6 layers, 6 heads, dim=192) -> [B, 1030, 192]
#     -> take CLS token [:, 0] -> [B, 192]
#     -> Dropout(0.3) -> Linear(192, num_classes)

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# ── Drop Path (Stochastic Depth) ─────────────────────────────────────────────
class DropPath(nn.Module):
    """Stochastic Depth: drops entire residual path per sample during training."""
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor = torch.floor(random_tensor + keep_prob)
        return x / keep_prob * random_tensor


# ── MultiScalePatchEmbed ──────────────────────────────────────────────────────
class MultiScalePatchEmbed(nn.Module):
    """
    Multi-scale patch embedding with per-scale embedding dimensions.
    Confirmed from: src/models/maizeformerx/multi_scale_patch_embed.py

    patch_sizes = [8, 16, 32], strides = [8, 16, 32]
    embed_dims  = [64, 96, 128]  -- each scale has its own dimension

    Each branch: Conv2d(3, embed_dim_i, kernel_size=ps_i, stride=ps_i) + LayerNorm

    Output: list of 3 tensors
      [B, 784, 64]  (scale 8:  28x28 patches)
      [B, 196, 96]  (scale 16: 14x14 patches)
      [B,  49, 128] (scale 32: 7x7 patches)
    """
    def __init__(self, img_size=224, patch_sizes=[8, 16, 32], embed_dims=[64, 96, 128]):
        super().__init__()
        assert len(patch_sizes) == len(embed_dims)
        self.scales = nn.ModuleList()
        self.norms  = nn.ModuleList()
        self.num_patches_per_scale = []

        for ps, ed in zip(patch_sizes, embed_dims):
            self.scales.append(
                nn.Conv2d(3, ed, kernel_size=ps, stride=ps)
            )
            self.norms.append(nn.LayerNorm(ed))
            self.num_patches_per_scale.append((img_size // ps) ** 2)

    def forward(self, x):
        """x: [B, 3, H, W] -> list of [B, Ni, Di]"""
        outputs = []
        for conv, norm in zip(self.scales, self.norms):
            t = conv(x)                              # [B, Di, Hi, Wi]
            B, D, H, W = t.shape
            t = t.flatten(2).transpose(1, 2)         # [B, Ni, Di]
            t = norm(t)
            outputs.append(t)
        return outputs


# ── CSAF (Cross-Scale Attention Fusion) ──────────────────────────────────────
class CSAF(nn.Module):
    """
    Cross-Scale Attention Fusion.
    Confirmed from: src/models/maizeformerx/csaf.py

    Input:  list [[B, N1, 64], [B, N2, 96], [B, N3, 128]]
    Steps:
      1. Project each scale to fusion_dim=192 via separate Linear layers:
         Linear(64->192), Linear(96->192), Linear(128->192)
      2. Concatenate all projected tokens: [B, N1+N2+N3, 192] = [B, 1029, 192]
      3. Self-attention over all 1029 tokens (num_heads=4)
      4. Residual connection + LayerNorm: output = LayerNorm(concat + attn_out)
    Output: [B, 1029, 192]
    """
    def __init__(self, in_dims=[64, 96, 128], fusion_dim=192,
                 num_heads=4, attn_dropout=0.1, proj_dropout=0.1):
        super().__init__()
        self.fusion_dim = fusion_dim

        # Per-scale linear projection to fusion_dim
        self.projections = nn.ModuleList([
            nn.Linear(d, fusion_dim) for d in in_dims
        ])

        # Self-attention over all concatenated tokens
        self.self_attn = nn.MultiheadAttention(
            embed_dim=fusion_dim,
            num_heads=num_heads,
            dropout=attn_dropout,
            batch_first=True
        )
        self.proj_drop = nn.Dropout(proj_dropout)
        self.norm      = nn.LayerNorm(fusion_dim)

    def forward(self, scale_tokens):
        """
        scale_tokens: list of [B, Ni, Di] tensors (3 scales)
        Returns: [B, N1+N2+N3, fusion_dim] = [B, 1029, 192]
        """
        # 1. Project each scale to fusion_dim
        projected = [proj(t) for proj, t in zip(self.projections, scale_tokens)]

        # 2. Concatenate: [B, 1029, 192]
        x = torch.cat(projected, dim=1)

        # 3. Self-attention + dropout
        attn_out, _ = self.self_attn(x, x, x)
        attn_out = self.proj_drop(attn_out)

        # 4. Residual + LayerNorm
        out = self.norm(x + attn_out)
        return out


# ── Transformer Encoder Layer ────────────────────────────────────────────────
class TransformerEncoderLayer(nn.Module):
    """
    Standard Transformer Encoder layer with Pre-Norm and DropPath.
    Confirmed from: src/models/maizeformerx/encoder.py

    dim=192, num_heads=6, mlp_ratio=4.0, qkv_bias=True
    Pre-norm: LayerNorm -> MHSA -> DropPath -> Residual -> LayerNorm -> MLP -> DropPath -> Residual
    """
    def __init__(self, dim=192, num_heads=6, mlp_ratio=4.0,
                 attn_dropout=0.1, proj_dropout=0.1, drop_path_rate=0.0,
                 qkv_bias=True):
        super().__init__()
        mlp_hidden = int(dim * mlp_ratio)

        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=attn_dropout,
            bias=qkv_bias,
            batch_first=True
        )
        self.proj_drop1 = nn.Dropout(proj_dropout)
        self.drop_path1 = DropPath(drop_path_rate)

        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(proj_dropout),
            nn.Linear(mlp_hidden, dim),
            nn.Dropout(proj_dropout)
        )
        self.drop_path2 = DropPath(drop_path_rate)

    def forward(self, x):
        # Pre-norm MHSA with DropPath
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + self.drop_path1(self.proj_drop1(attn_out))
        # Pre-norm MLP with DropPath
        x = x + self.drop_path2(self.mlp(self.norm2(x)))
        return x


# ── MaizeFormerX ─────────────────────────────────────────────────────────────
class MaizeFormerX(nn.Module):
    """
    MaizeFormerX: Lightweight Vision Transformer with Cross-Scale Attention Fusion.
    Architecture confirmed from: github.com/rezaul-h/MaizeFormerX/

    Flow:
      Input [B, 3, 224, 224]
        -> MultiScalePatchEmbed -> [[B,784,64], [B,196,96], [B,49,128]]
        -> CSAF (project + concat + self-attn) -> [B, 1029, 192]
        -> prepend CLS token -> [B, 1030, 192]
        -> add positional embedding
        -> TransformerEncoder (6 layers, 6 heads, drop_path linear 0->0.1) -> [B, 1030, 192]
        -> take CLS token [:, 0] -> [B, 192]
        -> Dropout(0.3) -> Linear(192, num_classes)
    """
    def __init__(
        self,
        img_size=224,
        patch_sizes=[8, 16, 32],
        embed_dims=[64, 96, 128],
        fusion_dim=192,
        num_layers=6,
        num_heads_encoder=6,
        num_heads_csaf=4,
        mlp_ratio=4.0,
        attn_dropout=0.1,
        proj_dropout=0.1,
        drop_path=0.1,       # max drop_path — linearly scheduled 0 -> drop_path
        dropout=0.3,         # classifier head dropout
        num_classes=4
    ):
        super().__init__()
        self.fusion_dim = fusion_dim

        # 1. Multi-scale patch embedding
        self.patch_embed = MultiScalePatchEmbed(
            img_size=img_size,
            patch_sizes=patch_sizes,
            embed_dims=embed_dims
        )

        # 2. CSAF
        self.csaf = CSAF(
            in_dims=embed_dims,
            fusion_dim=fusion_dim,
            num_heads=num_heads_csaf,
            attn_dropout=attn_dropout,
            proj_dropout=proj_dropout
        )

        # 3. CLS token (learnable parameter)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, fusion_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # Positional embedding for CLS + all patch tokens
        # N_total = N1 + N2 + N3 + 1 = 784 + 196 + 49 + 1 = 1030
        n_total = sum(self.patch_embed.num_patches_per_scale) + 1
        self.pos_embed = nn.Parameter(torch.zeros(1, n_total, fusion_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # 4. Transformer encoder with linear drop_path schedule
        # dpr[0]=0.0, dpr[5]=0.1 (linearly spaced)
        dpr = [x.item() for x in torch.linspace(0, drop_path, num_layers)]
        self.transformer = nn.ModuleList([
            TransformerEncoderLayer(
                dim=fusion_dim,
                num_heads=num_heads_encoder,
                mlp_ratio=mlp_ratio,
                attn_dropout=attn_dropout,
                proj_dropout=proj_dropout,
                drop_path_rate=dpr[i],
                qkv_bias=True
            )
            for i in range(num_layers)
        ])
        self.norm = nn.LayerNorm(fusion_dim)

        # 5. Classifier: CLS token -> Dropout -> Linear
        self.dropout = nn.Dropout(dropout)
        self.head    = nn.Linear(fusion_dim, num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        B = x.shape[0]

        # 1. Multi-scale patch embedding: list of [B, Ni, Di]
        scale_tokens = self.patch_embed(x)

        # 2. CSAF: project + concat + self-attn -> [B, 1029, 192]
        fused = self.csaf(scale_tokens)   # [B, N1+N2+N3, 192]

        # 3. Prepend CLS token: [B, 1030, 192]
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, fused], dim=1)

        # 4. Add positional embedding
        x = x + self.pos_embed

        # 5. Transformer encoder
        for layer in self.transformer:
            x = layer(x)
        x = self.norm(x)

        # 6. Take CLS token output, classify
        cls_out = x[:, 0]              # [B, 192]
        cls_out = self.dropout(cls_out)
        logits  = self.head(cls_out)   # [B, num_classes]
        return logits


# ── Quick sanity check ────────────────────────────────────────────────────────
_test_model = MaizeFormerX(
    img_size=IMG_SIZE,
    patch_sizes=PATCH_SIZES,
    embed_dims=EMBED_DIMS,
    fusion_dim=FUSION_DIM,
    num_layers=NUM_LAYERS,
    num_heads_encoder=NUM_HEADS_ENCODER,
    num_heads_csaf=NUM_HEADS_CSAF,
    mlp_ratio=MLP_RATIO,
    attn_dropout=ATTN_DROPOUT,
    proj_dropout=PROJ_DROPOUT,
    drop_path=DROP_PATH,
    dropout=DROPOUT,
    num_classes=4   # placeholder — NUM_CLASSES auto-detected later
)
_dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
_out   = _test_model(_dummy)
print(f"Output shape: {_out.shape}  (expected: [1, 4])")

_emb = _test_model.patch_embed
for ps, ed, n in zip(PATCH_SIZES, EMBED_DIMS, _emb.num_patches_per_scale):
    print(f"  patch_size={ps:2d} -> {n:4d} tokens, embed_dim={ed}")
print(f"  CSAF concat -> {sum(_emb.num_patches_per_scale)} tokens, fusion_dim={FUSION_DIM}")
print(f"  + CLS token -> {sum(_emb.num_patches_per_scale)+1} tokens total in encoder")

del _test_model, _dummy, _out
print("MaizeFormerX architecture defined successfully.")


## BLOCK 6: Model Summary

In [ ]:
# ── Model Summary (torchinfo) ─────────────────────────────────────────────────
# NUM_CLASSES not yet known here (auto-detected in BLOCK 9).
# Using placeholder=4 for summary. Actual model instantiation uses real NUM_CLASSES.
_summary_model = MaizeFormerX(
    img_size=IMG_SIZE,
    patch_sizes=PATCH_SIZES,
    embed_dims=EMBED_DIMS,
    fusion_dim=FUSION_DIM,
    num_layers=NUM_LAYERS,
    num_heads_encoder=NUM_HEADS_ENCODER,
    num_heads_csaf=NUM_HEADS_CSAF,
    mlp_ratio=MLP_RATIO,
    attn_dropout=ATTN_DROPOUT,
    proj_dropout=PROJ_DROPOUT,
    drop_path=DROP_PATH,
    dropout=DROPOUT,
    num_classes=4   # placeholder
)

model_summary = summary(
    _summary_model,
    input_size=(1, 3, IMG_SIZE, IMG_SIZE),
    col_names=["output_size", "num_params"],
    verbose=1
)
del _summary_model


## BLOCK 7: Training Utilities

In [ ]:

# ── Training Utilities ────────────────────────────────────────────────────────

def compute_class_weights(train_dataset):
    if hasattr(train_dataset, 'targets'):
        labels = np.array(train_dataset.targets)
    elif hasattr(train_dataset, 'dataset'):
        targets = np.array(train_dataset.dataset.targets)
        labels  = targets[train_dataset.indices]
    else:
        labels = np.array([train_dataset[i][1] for i in range(len(train_dataset))])

    unique, counts = np.unique(labels, return_counts=True)
    imbalance_ratio = counts.max() / counts.min()
    print(f"Class distribution: {dict(zip(unique.tolist(), counts.tolist()))}")
    print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}")

    if imbalance_ratio >= 1.5:
        print("[WARNING] Class imbalance detected (ratio >= 1.5). Using weighted CrossEntropyLoss.")
        weights = compute_class_weight('balanced', classes=unique, y=labels)
        return torch.tensor(weights, dtype=torch.float32), True
    else:
        print("[OK] No significant class imbalance. Using standard CrossEntropyLoss.")
        return None, False


_AMP_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_AMP_ENABLED = torch.cuda.is_available()


def train_with_checkpoint(
    model, train_loader, val_loader, optimizer, scheduler,
    criterion, max_epochs, patience, checkpoint_path, warmup_steps=300
):
    """
    Training loop with AMP, warmup, cosine LR, best checkpoint, early stopping.
    DataParallel-aware: saves model.module.state_dict() if wrapped.
    """
    model = model.to(device)
    scaler = GradScaler(_AMP_DEVICE, enabled=_AMP_ENABLED)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0.0
    patience_counter = 0
    global_step = 0

    for epoch in range(max_epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            if global_step < warmup_steps:
                lr_scale = (global_step + 1) / warmup_steps
                for pg in optimizer.param_groups:
                    pg['lr'] = LR * lr_scale

            optimizer.zero_grad()
            with autocast(_AMP_DEVICE, enabled=_AMP_ENABLED):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss    += loss.item() * images.size(0)
            preds          = outputs.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total   += images.size(0)
            global_step   += 1

        if global_step >= warmup_steps:
            scheduler.step()

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                with autocast(_AMP_DEVICE, enabled=_AMP_ENABLED):
                    outputs = model(images)
                    loss    = criterion(outputs, labels)
                val_loss    += loss.item() * images.size(0)
                preds        = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total   += images.size(0)

        avg_train_loss = train_loss / train_total
        avg_val_loss   = val_loss   / val_total
        avg_train_acc  = train_correct / train_total
        avg_val_acc    = val_correct   / val_total

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_acc'].append(avg_train_acc)
        history['val_acc'].append(avg_val_acc)

        print(f"Epoch [{epoch+1:02d}/{max_epochs}] "
              f"train_loss={avg_train_loss:.4f} val_loss={avg_val_loss:.4f} "
              f"train_acc={avg_train_acc:.4f} val_acc={avg_val_acc:.4f}")

        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            # DataParallel-safe checkpoint save (Rule 10 + Rule 14)
            _state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(_state, checkpoint_path)
            patience_counter = 0
            print(f"  [Checkpoint] Best val_acc={best_val_acc:.4f} saved to {checkpoint_path}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  [EarlyStopping] No improvement for {patience} epochs. Stopping.")
                break

    # Reload best checkpoint (DataParallel-safe)
    _target = model.module if isinstance(model, nn.DataParallel) else model
    _target.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"\nReloaded best model (val_acc={best_val_acc:.4f}) from {checkpoint_path}")
    return model, history


def evaluate_model(model, test_loader, class_names):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            with autocast(_AMP_DEVICE, enabled=_AMP_ENABLED):
                outputs = model(images)
            preds   = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc    = accuracy_score(all_labels, all_preds)
    f1     = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    # Count params on .module if DataParallel
    _m = model.module if isinstance(model, nn.DataParallel) else model
    total_params = sum(p.numel() for p in _m.parameters())

    print(f"\n{'='*60}")
    print(f"Test Accuracy : {acc:.4f}")
    print(f"F1-Score (Macro): {f1:.4f}")
    print(f"Recall (Macro): {recall:.4f}")
    print(f"Total Params  : {total_params:,}")
    print(f"{'='*60}\n")

    print("Classification Report (per class):")
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

    metrics = {
        "model_name"  : "MaizeFormerX",
        "accuracy"    : acc,
        "f1_macro"    : f1,
        "recall_macro": recall,
        "total_params": total_params
    }
    return metrics, (all_preds, all_labels)


print("Training utilities defined (AMP + DataParallel-safe checkpoint).")


## BLOCK 8: Plot Utilities

In [ ]:
# ── Plot Utilities ────────────────────────────────────────────────────────────

def plot_learning_curve(history, title, save_path):
    """Plot and save train/val loss and accuracy curves."""
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, history['train_loss'], 'b-o', markersize=4, label='Train Loss')
    ax1.plot(epochs, history['val_loss'],   'r-o', markersize=4, label='Val Loss')
    ax1.set_title(f'{title} — Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, history['train_acc'], 'b-o', markersize=4, label='Train Acc')
    ax2.plot(epochs, history['val_acc'],   'r-o', markersize=4, label='Val Acc')
    ax2.set_title(f'{title} — Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Learning curve saved to {save_path}")


def plot_confusion_matrix(all_labels, all_preds, class_names, title, save_path):
    """Plot and save confusion matrix."""
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(max(8, len(class_names)), max(6, len(class_names))))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names, ax=ax
    )
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_title(f'{title} — Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Confusion matrix saved to {save_path}")


print("Plot utilities defined.")

## BLOCK 9: Dataset Exploration & Class Imbalance Check

In [ ]:
# ── Dataset Exploration ───────────────────────────────────────────────────────
# Use baseline transform to count samples without augmentation
_transform_explore = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

_train_explore, _val_explore, _test_explore = make_splits(
    _transform_explore, _transform_explore
)

print(f"Train: {len(_train_explore)} | Val: {len(_val_explore)} | Test: {len(_test_explore)}")

# Auto-detect class names
class_names = get_class_names(_train_explore)
print(f"Class names: {class_names}")
print(f"NUM_CLASSES confirmed: {len(class_names)}")
NUM_CLASSES = len(class_names)  # update in case auto-detection differs

In [ ]:
# ── Dataset Distribution Bar Chart ────────────────────────────────────────────
# Count samples per class in train set
if hasattr(_train_explore, 'targets'):
    train_labels = np.array(_train_explore.targets)
elif hasattr(_train_explore, 'dataset'):
    _all_targets = np.array(_train_explore.dataset.targets)
    train_labels = _all_targets[_train_explore.indices]
else:
    train_labels = np.array([_train_explore[i][1] for i in range(len(_train_explore))])

unique_cls, counts = np.unique(train_labels, return_counts=True)
cls_counts = dict(zip([class_names[i] for i in unique_cls], counts))

fig, ax = plt.subplots(figsize=(max(8, len(class_names)*1.2), 5))
bars = ax.bar(cls_counts.keys(), cls_counts.values(), color='steelblue', edgecolor='white')
for bar, count in zip(bars, cls_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontsize=10)
ax.set_title('Training Set — Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Sample Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('/kaggle/working/dataset_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

imbalance_ratio = counts.max() / counts.min()
print(f"Imbalance ratio: {imbalance_ratio:.2f}")
print(f"Class counts: {cls_counts}")

---
# SCENARIO 1: Baseline (No Augmentation)

**Training transform:** Resize(224) + ToTensor + Normalize (ImageNet stats)  
**Val/Test transform:** Same as training (no aug)

Paper Rule 3 (DUAL EXPERIMENT): Training transform = ONLY Resize + ToTensor + Normalize

## S1 — BLOCK 1: Transforms

In [ ]:
# ── Scenario 1 Transforms (Baseline — NO augmentation) ───────────────────────
transform_baseline = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# Validation/Test always use baseline (no aug)
transform_eval = transform_baseline

print("Scenario 1 transforms defined (Resize + ToTensor + Normalize — NO augmentation)")

## S1 — BLOCK 2: Data Loaders

In [ ]:

# ── Scenario 1 Data Loaders ───────────────────────────────────────────────────
train_ds_s1, val_ds_s1, test_ds_s1 = make_splits(transform_baseline, transform_eval)

print(f"Train: {len(train_ds_s1)} | Val: {len(val_ds_s1)} | Test: {len(test_ds_s1)}")

class_weights_s1, use_weighting_s1 = compute_class_weights(train_ds_s1)

# num_workers=4 for Kaggle T4×2 (4 vCPU available, Rule 13)
train_loader_s1 = DataLoader(
    train_ds_s1, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True
)
val_loader_s1 = DataLoader(
    val_ds_s1, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)
test_loader_s1 = DataLoader(
    test_ds_s1, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

print(f"DataLoaders created (num_workers=4). Batches — Train: {len(train_loader_s1)} | Val: {len(val_loader_s1)} | Test: {len(test_loader_s1)}")


## S1 — BLOCK 3: Build Model & Optimizer

In [ ]:

# ── Scenario 1 — Model, Loss, Optimizer, Scheduler ───────────────────────────
model_s1 = MaizeFormerX(
    img_size=IMG_SIZE, patch_sizes=PATCH_SIZES, embed_dims=EMBED_DIMS,
    fusion_dim=FUSION_DIM, num_layers=NUM_LAYERS,
    num_heads_encoder=NUM_HEADS_ENCODER, num_heads_csaf=NUM_HEADS_CSAF,
    mlp_ratio=MLP_RATIO, attn_dropout=ATTN_DROPOUT, proj_dropout=PROJ_DROPOUT,
    drop_path=DROP_PATH, dropout=DROPOUT, num_classes=NUM_CLASSES
).to(device)

# Multi-GPU: wrap with DataParallel if 2 GPUs available (Rule 8)
if n_gpus > 1:
    model_s1 = nn.DataParallel(model_s1)
    print(f"[Multi-GPU] DataParallel on {n_gpus} GPUs")

# Loss function
if use_weighting_s1 and class_weights_s1 is not None:
    criterion_s1 = nn.CrossEntropyLoss(weight=class_weights_s1.to(device))
    print("Using weighted CrossEntropyLoss.")
else:
    criterion_s1 = nn.CrossEntropyLoss()
    print("Using standard CrossEntropyLoss.")

optimizer_s1 = torch.optim.Adam(model_s1.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler_s1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_s1, T_max=MAX_EPOCHS)

CHECKPOINT_S1 = '/kaggle/working/best_model_s1.pth'
_m_s1 = model_s1.module if isinstance(model_s1, nn.DataParallel) else model_s1
print(f"Model S1 initialized. fusion_dim={FUSION_DIM}, num_layers={NUM_LAYERS}")
print(f"  num_heads_encoder={NUM_HEADS_ENCODER}, num_heads_csaf={NUM_HEADS_CSAF}")
print(f"  Checkpoint: {CHECKPOINT_S1}")


## S1 — BLOCK 4: Train

In [ ]:
# ── Scenario 1 — Training ─────────────────────────────────────────────────────
print("=" * 60)
print("SCENARIO 1: Training — Baseline (No Augmentation)")
print("=" * 60)

model_s1, history_s1 = train_with_checkpoint(
    model=model_s1,
    train_loader=train_loader_s1,
    val_loader=val_loader_s1,
    optimizer=optimizer_s1,
    scheduler=scheduler_s1,
    criterion=criterion_s1,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=CHECKPOINT_S1,
    warmup_steps=WARMUP_STEPS
)

## S1 — BLOCK 5: Classification Report

In [ ]:
# ── Scenario 1 — Evaluation & Classification Report ──────────────────────────
print("SCENARIO 1 — Evaluation on Test Set:")
baseline_metrics, (s1_preds, s1_labels) = evaluate_model(
    model_s1, test_loader_s1, class_names
)

## S1 — BLOCK 6: Confusion Matrix

In [ ]:
# ── Scenario 1 — Confusion Matrix ────────────────────────────────────────────
BASELINE_CONFUSION_MATRIX_PATH = '/kaggle/working/baseline_confusion_matrix.png'

plot_confusion_matrix(
    s1_labels, s1_preds, class_names,
    title='MaizeFormerX — Scenario 1 (No Aug)',
    save_path=BASELINE_CONFUSION_MATRIX_PATH
)

## S1 — BLOCK 7: Learning Curve

In [ ]:
# ── Scenario 1 — Learning Curve ───────────────────────────────────────────────
BASELINE_LEARNING_CURVE_PATH = '/kaggle/working/baseline_learning_curve.png'

plot_learning_curve(
    history_s1,
    title='MaizeFormerX — Scenario 1 (No Aug)',
    save_path=BASELINE_LEARNING_CURVE_PATH
)

print(f"\nScenario 1 complete.")
print(f"  Accuracy : {baseline_metrics['accuracy']:.4f}")
print(f"  F1 Macro : {baseline_metrics['f1_macro']:.4f}")
print(f"  Recall   : {baseline_metrics['recall_macro']:.4f}")
print(f"  Params   : {baseline_metrics['total_params']:,}")

## S1 — BLOCK 8: Clear GPU Cache Before Scenario 2

In [ ]:
# ── Clear GPU Memory ──────────────────────────────────────────────────────────
# Rule 13: torch.cuda.empty_cache() after Scenario 1, before Scenario 2
del model_s1, optimizer_s1, scheduler_s1, criterion_s1
del train_loader_s1, val_loader_s1, test_loader_s1
del train_ds_s1, val_ds_s1, test_ds_s1

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU cache cleared.")

import gc
gc.collect()
print("Ready for Scenario 2.")

---
# SCENARIO 2: Augmented

**Training transform:** Resize + RandomRotation(±30°) + RandomResizedCrop(scale=0.9–1.1) + RandomHorizontalFlip + ColorJitter + ToTensor + Normalize  
**Val/Test transform:** Same as baseline (no aug)

Augmentation values from paper (Table 2, Dataverse dataset config):
- Horizontal Flip: p=0.5
- Rotation: ±30°
- Zoom/Scale: 0.9–1.1
- Brightness: ±20%

## S2 — BLOCK 1: Transforms

In [ ]:
# ── Scenario 2 Transforms (Augmented) ────────────────────────────────────────
# Augmentation from paper Table 2 (Dataverse dataset values):
#   Horizontal Flip:  p=0.5
#   Rotation:         ±30 degrees
#   Zoom/Scale:       0.9 to 1.1 (scale factor)
#   Brightness:       ±20% (ColorJitter brightness=0.2)

transform_augmented = transforms.Compose([
    transforms.Resize((int(IMG_SIZE * 1.1), int(IMG_SIZE * 1.1))),
    transforms.RandomRotation(degrees=30),                   # paper: ±30°
    transforms.RandomResizedCrop(
        size=IMG_SIZE,
        scale=(0.9, 1.0),                                    # paper: zoom 0.9–1.1
        ratio=(0.75, 1.333)
    ),
    transforms.RandomHorizontalFlip(p=0.5),                  # paper: p=0.5
    transforms.ColorJitter(
        brightness=0.2,                                      # paper: ±20% brightness
        contrast=0.0,
        saturation=0.0,
        hue=0.0
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# Val/Test always use baseline (no aug) — Rule 3
transform_eval_s2 = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

print("Scenario 2 transforms defined:")
print("  Train: Resize(246) + RandomRotation(30) + RandomResizedCrop(224, scale=0.9-1.0)")
print("         + RandomHorizontalFlip(0.5) + ColorJitter(brightness=0.2)")
print("  Val/Test: Resize(224) + ToTensor + Normalize (no aug)")

## S2 — BLOCK 2: Data Loaders

In [ ]:

# ── Scenario 2 Data Loaders ───────────────────────────────────────────────────
train_ds_s2, val_ds_s2, test_ds_s2 = make_splits(transform_augmented, transform_eval_s2)

print(f"Train: {len(train_ds_s2)} | Val: {len(val_ds_s2)} | Test: {len(test_ds_s2)}")

class_weights_s2, use_weighting_s2 = compute_class_weights(train_ds_s2)

# num_workers=4 for Kaggle T4×2 (4 vCPU available, Rule 13)
train_loader_s2 = DataLoader(
    train_ds_s2, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True
)
val_loader_s2 = DataLoader(
    val_ds_s2, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)
test_loader_s2 = DataLoader(
    test_ds_s2, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

print(f"DataLoaders created (num_workers=4). Batches — Train: {len(train_loader_s2)} | Val: {len(val_loader_s2)} | Test: {len(test_loader_s2)}")


## S2 — BLOCK 3: Build Model & Optimizer

In [ ]:

# ── Scenario 2 — Model, Loss, Optimizer, Scheduler ───────────────────────────
model_s2 = MaizeFormerX(
    img_size=IMG_SIZE, patch_sizes=PATCH_SIZES, embed_dims=EMBED_DIMS,
    fusion_dim=FUSION_DIM, num_layers=NUM_LAYERS,
    num_heads_encoder=NUM_HEADS_ENCODER, num_heads_csaf=NUM_HEADS_CSAF,
    mlp_ratio=MLP_RATIO, attn_dropout=ATTN_DROPOUT, proj_dropout=PROJ_DROPOUT,
    drop_path=DROP_PATH, dropout=DROPOUT, num_classes=NUM_CLASSES
).to(device)

# Multi-GPU: wrap with DataParallel if 2 GPUs available (Rule 8)
if n_gpus > 1:
    model_s2 = nn.DataParallel(model_s2)
    print(f"[Multi-GPU] DataParallel on {n_gpus} GPUs")

if use_weighting_s2 and class_weights_s2 is not None:
    criterion_s2 = nn.CrossEntropyLoss(weight=class_weights_s2.to(device))
    print("Using weighted CrossEntropyLoss.")
else:
    criterion_s2 = nn.CrossEntropyLoss()
    print("Using standard CrossEntropyLoss.")

optimizer_s2 = torch.optim.Adam(model_s2.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler_s2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_s2, T_max=MAX_EPOCHS)

CHECKPOINT_S2 = '/kaggle/working/best_model_s2.pth'
print(f"Model S2 initialized. Checkpoint: {CHECKPOINT_S2}")


## S2 — BLOCK 4: Train

In [ ]:
# ── Scenario 2 — Training ─────────────────────────────────────────────────────
print("=" * 60)
print("SCENARIO 2: Training — Augmented")
print("=" * 60)

model_s2, history_s2 = train_with_checkpoint(
    model=model_s2,
    train_loader=train_loader_s2,
    val_loader=val_loader_s2,
    optimizer=optimizer_s2,
    scheduler=scheduler_s2,
    criterion=criterion_s2,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=CHECKPOINT_S2,
    warmup_steps=WARMUP_STEPS
)

## S2 — BLOCK 5: Classification Report

In [ ]:
# ── Scenario 2 — Evaluation & Classification Report ──────────────────────────
print("SCENARIO 2 — Evaluation on Test Set:")
aug_metrics, (s2_preds, s2_labels) = evaluate_model(
    model_s2, test_loader_s2, class_names
)

## S2 — BLOCK 6: Confusion Matrix

In [ ]:
# ── Scenario 2 — Confusion Matrix ────────────────────────────────────────────
AUG_CONFUSION_MATRIX_PATH = '/kaggle/working/aug_confusion_matrix.png'

plot_confusion_matrix(
    s2_labels, s2_preds, class_names,
    title='MaizeFormerX — Scenario 2 (Augmented)',
    save_path=AUG_CONFUSION_MATRIX_PATH
)

## S2 — BLOCK 7: Learning Curve

In [ ]:
# ── Scenario 2 — Learning Curve ───────────────────────────────────────────────
AUG_LEARNING_CURVE_PATH = '/kaggle/working/aug_learning_curve.png'

plot_learning_curve(
    history_s2,
    title='MaizeFormerX — Scenario 2 (Augmented)',
    save_path=AUG_LEARNING_CURVE_PATH
)

print(f"\nScenario 2 complete.")
print(f"  Accuracy : {aug_metrics['accuracy']:.4f}")
print(f"  F1 Macro : {aug_metrics['f1_macro']:.4f}")
print(f"  Recall   : {aug_metrics['recall_macro']:.4f}")
print(f"  Params   : {aug_metrics['total_params']:,}")

---
# GRAD-CAM VISUALIZATION

Paper Section: Explainability analysis with Grad-CAM at the CSAF output layer.  
Paper shows Grad-CAM heatmaps per class on all three datasets (Figures 19–23).

In [ ]:
# ── Grad-CAM Visualization ────────────────────────────────────────────────────
# Paper Fig 19-23: Grad-CAM explainability analysis.
#
# Architecture: CLS token (index 0) + 1029 patch tokens = 1030 total.
# Token layout in encoder input: [CLS | N1=784 fine | N2=196 mid | N3=49 coarse]
#   - Fine scale (patch_size=8): tokens [1 : 785]   -> reshape to [28, 28]
#   - Mid scale  (patch_size=16): tokens [785 : 981] -> reshape to [14, 14]
#   - Coarse scale (patch_size=32): tokens [981: 1030] -> reshape to [7, 7]
#
# Target layer: model.norm (final LayerNorm applied to all 1030 tokens).
# reshape_transform: skip CLS (index 0), take fine-scale tokens [1:785], reshape to [B, 192, 28, 28].

def get_test_samples(test_loader, num_per_class=1):
    """Get one sample per class from test loader."""
    samples = {cls_idx: None for cls_idx in range(NUM_CLASSES)}
    for images, labels in test_loader:
        for i in range(len(labels)):
            cls = labels[i].item()
            if samples[cls] is None:
                samples[cls] = (images[i:i+1], labels[i:i+1])
        if all(v is not None for v in samples.values()):
            break
    return samples


def reshape_transform_mspe(tensor):
    """
    tensor: [B, 1030, 192] (all tokens after encoder + norm)
    Skip CLS token (index 0).
    Take fine-scale tokens (index 1..784) -> reshape to spatial [B, 192, 28, 28].
    GradCAM needs 4D [B, C, H, W] output.
    """
    fine_tokens = tensor[:, 1:785, :]      # [B, 784, 192]
    B, N, D = fine_tokens.shape
    h = w = int(math.sqrt(N))             # 28
    result = fine_tokens.reshape(B, h, w, D).permute(0, 3, 1, 2)  # [B, 192, 28, 28]
    return result


try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

    # DataParallel-safe: GradCAM must use the inner model, not the DP wrapper
    _model_for_cam = model_s2.module if isinstance(model_s2, nn.DataParallel) else model_s2
    _model_for_cam.eval()
    target_layer = _model_for_cam.norm   # final LayerNorm after transformer stack

    cam = GradCAM(
        model=_model_for_cam,
        target_layers=[target_layer],
        reshape_transform=reshape_transform_mspe
    )

    samples = get_test_samples(test_loader_s2)

    n_cols = min(NUM_CLASSES, 6)
    n_rows = math.ceil(NUM_CLASSES / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 4*n_rows))
    axes_flat = np.array(axes).flatten() if (n_rows * n_cols) > 1 else np.array([axes]).flatten()

    for idx, (cls_idx, sample) in enumerate(samples.items()):
        if sample is None:
            continue
        img_tensor, label = sample
        img_tensor = img_tensor.to(device)

        grayscale_cam = cam(
            input_tensor=img_tensor,
            targets=[ClassifierOutputTarget(cls_idx)]
        )
        grayscale_cam = grayscale_cam[0]

        # Inverse normalize for visualization
        img_vis = img_tensor[0].cpu().clone()
        for c in range(3):
            img_vis[c] = img_vis[c] * STD[c] + MEAN[c]
        img_np = img_vis.permute(1, 2, 0).numpy()
        img_np = np.clip(img_np, 0, 1)

        cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

        ax = axes_flat[idx]
        ax.imshow(cam_image)
        ax.set_title(f"Class: {class_names[cls_idx]}", fontsize=10)
        ax.axis('off')

    for idx in range(NUM_CLASSES, len(axes_flat)):
        axes_flat[idx].axis('off')

    plt.suptitle('MaizeFormerX — Grad-CAM Visualization (per class)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/kaggle/working/gradcam_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    print("Grad-CAM visualization saved.")

except Exception as e:
    print(f"[WARNING] Grad-CAM skipped: {e}")
    print("Grad-CAM is optional — results_summary.xlsx is still complete.")


---
# RESULTS SUMMARY

In [ ]:
# ── Results Summary ───────────────────────────────────────────────────────────
print("=" * 70)
print("FINAL RESULTS SUMMARY — MaizeFormerX")
print("=" * 70)
print(f"{'Metric':<20} {'Scenario 1 (No Aug)':>22} {'Scenario 2 (Augmented)':>24}")
print("-" * 70)
print(f"{'Accuracy':<20} {baseline_metrics['accuracy']:>22.4f} {aug_metrics['accuracy']:>24.4f}")
print(f"{'F1-Score (Macro)':<20} {baseline_metrics['f1_macro']:>22.4f} {aug_metrics['f1_macro']:>24.4f}")
print(f"{'Recall (Macro)':<20} {baseline_metrics['recall_macro']:>22.4f} {aug_metrics['recall_macro']:>24.4f}")
print(f"{'Total Params':<20} {baseline_metrics['total_params']:>22,} {aug_metrics['total_params']:>24,}")
print("=" * 70)

print("\nPaper reference results (on Dataverse, x6 augmentation):")
print("  Accuracy: 97.8% | F1: ~98.6% (macro) | MCC: 95.3%")
print("  Note: Paper results are on their own dataset, NOT directly comparable.")

---
# EXCEL EXPORT

In [ ]:
# ── Install openpyxl (if not already done) ────────────────────────────────────
!pip install -q openpyxl Pillow

In [ ]:
# ── Export results_summary.xlsx ───────────────────────────────────────────────
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter

EXCEL_OUTPUT_PATH = "/kaggle/working/results_summary.xlsx"
MODEL_NAME        = "MaizeFormerX"

HEADERS    = ["STT", "Ten Model", "Accuracy", "F1-score", "Recall",
              "Learning Curve", "Confusion Matrix", "Parameters"]
COL_WIDTHS = [6, 35, 12, 12, 12, 38, 38, 18]
IMG_ROW_HEIGHT = 200


def style_header_row(ws):
    fill   = PatternFill("solid", fgColor="1F4E79")
    font   = Font(bold=True, color="FFFFFF", size=11)
    border = Border(
        bottom=Side(style="medium", color="FFFFFF"),
        right=Side(style="thin",   color="FFFFFF")
    )
    for col, header in enumerate(HEADERS, start=1):
        cell = ws.cell(row=1, column=col, value=header)
        cell.fill, cell.font, cell.border = fill, font, border
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)


def write_result_row(ws, row_num, stt, metrics, lc_path, cm_path):
    data = [
        stt,
        metrics["model_name"],
        round(metrics["accuracy"], 4),
        round(metrics["f1_macro"], 4),
        round(metrics["recall_macro"], 4),
        "",   # Learning Curve placeholder
        "",   # Confusion Matrix placeholder
        f"{metrics['total_params']:,}"
    ]
    for col, val in enumerate(data, start=1):
        cell = ws.cell(row=row_num, column=col, value=val)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    ws.row_dimensions[row_num].height = IMG_ROW_HEIGHT

    if os.path.exists(lc_path):
        img_lc = XLImage(lc_path)
        img_lc.width, img_lc.height = 280, 190
        ws.add_image(img_lc, f"F{row_num}")

    if os.path.exists(cm_path):
        img_cm = XLImage(cm_path)
        img_cm.width, img_cm.height = 280, 190
        ws.add_image(img_cm, f"G{row_num}")


def build_sheet(wb, sheet_name, metrics, lc_path, cm_path):
    ws = wb.create_sheet(title=sheet_name)
    for col, width in enumerate(COL_WIDTHS, start=1):
        ws.column_dimensions[get_column_letter(col)].width = width
    ws.row_dimensions[1].height = 30
    style_header_row(ws)
    write_result_row(ws, row_num=2, stt=1, metrics=metrics,
                     lc_path=lc_path, cm_path=cm_path)
    return ws


# Build workbook
wb = Workbook()
wb.remove(wb.active)

build_sheet(
    wb, "Baseline",
    metrics=baseline_metrics,
    lc_path=BASELINE_LEARNING_CURVE_PATH,
    cm_path=BASELINE_CONFUSION_MATRIX_PATH
)

build_sheet(
    wb, "Augmented",
    metrics=aug_metrics,
    lc_path=AUG_LEARNING_CURVE_PATH,
    cm_path=AUG_CONFUSION_MATRIX_PATH
)

wb.save(EXCEL_OUTPUT_PATH)
print(f"[Excel] Saved: {EXCEL_OUTPUT_PATH}")
print(f"  Baseline  -> Accuracy={baseline_metrics['accuracy']:.4f} | "
      f"F1={baseline_metrics['f1_macro']:.4f} | "
      f"Recall={baseline_metrics['recall_macro']:.4f} | "
      f"Params={baseline_metrics['total_params']:,}")
print(f"  Augmented -> Accuracy={aug_metrics['accuracy']:.4f} | "
      f"F1={aug_metrics['f1_macro']:.4f} | "
      f"Recall={aug_metrics['recall_macro']:.4f} | "
      f"Params={aug_metrics['total_params']:,}")

---
# OUTPUT FILES

After running this notebook on Kaggle, download from `/kaggle/working/`:

```
/kaggle/working/
├── results_summary.xlsx           <- Main result table (2 sheets, embedded images)
├── baseline_learning_curve.png
├── baseline_confusion_matrix.png
├── aug_learning_curve.png
├── aug_confusion_matrix.png
├── gradcam_visualization.png
├── dataset_distribution.png
├── best_model_s1.pth              <- Best weights Scenario 1
└── best_model_s2.pth              <- Best weights Scenario 2
```